# 01 - Data Preparation

**Dataset**: Online Retail II (UK-based online retailer, giftware, mostly wholesale)

**Source**: UCI Machine Learning Repository

**Scope**: 1,067,371 transaction lines, December 2009 to December 2011, 9 variables, GBP

---

## Business Context & Objectives

This notebook turns raw retail transactions into a reliable customer analytics dataset. It establishes the cleaning decisions that every downstream notebook inherits: relational modelling (02), customer segmentation (03), retention and churn (04), repurchase prediction (05).

Preparation is driven by the analyses that follow rather than by generic cleaning rules. Each anomaly is inspected against real rows before deciding whether it is removed, retained, or retained with a flag.

### Objectives

1. Explore the raw data (schema, types, missingness, volume) before applying any filter.
2. Identify and resolve retail-specific anomalies: cancellations, non-product stock codes, zero or negative prices, system test data.
3. Quantify the impact of every decision in rows removed and revenue affected.
4. Log each assumption so downstream notebooks inherit an auditable set of choices.

### Planned cleaning decisions

| Observation | Decision | Rationale |
|---|---|---|
| Missing Customer ID | Remove | Customer-level analyses require a unique customer identifier |
| Non-product stock codes | Remove | These lines are not product purchases |
| Cancelled invoices | Keep and flag | Returns carry behavioural information, handled differently per analysis |
| Negative quantities | Inspect before deciding | Most correspond to returns |
| Zero or negative prices | Inspect before deciding | May be gifts, adjustments or data-entry errors |

### Variables

| Type | Variables |
|---|---|
| Transaction | `Invoice`, `InvoiceDate`, `Quantity`, `Price` |
| Product | `StockCode`, `Description` |
|

# 0. Setup & Loading

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

RAW = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = PROC / "raw_concat.parquet"

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    source_info = f"Loaded dataset from Parquet cache (`{PARQUET_PATH.name}`)"
else:
    sheets = pd.read_excel(RAW, sheet_name=None)
    df = pd.concat(
        [sheet.assign(SourceSheet=name) for name, sheet in sheets.items()],
        ignore_index=True,
    )
    df.columns = df.columns.str.strip()

    # 'Invoice' mixes integers and cancellation codes ('C489449');
    # 'StockCode' mixes numeric codes and non-product codes (POST, DOT, M...);
    # 'Description' mixes a few numeric values among free text.
    # 'string' dtype preserves missing values as <NA> instead of literal "nan".
    text_cols = df.select_dtypes(include="object").columns
    df[text_cols] = df[text_cols].apply(lambda s: s.astype("string").str.strip())

    df.to_parquet(PARQUET_PATH, index=False)
    source_info = f"Parsed raw Excel sheets and cached to `{PARQUET_PATH.name}`"

display(Markdown(f"### Data Loading Completed\n* **Source:** {source_info}"))


### Data Loading Completed
* **Source:** Parsed raw Excel sheets and cached to `raw_concat.parquet`

# 1. Dataset Overview
## 1.1 Summary & Schema

In [2]:
sheets_list = ", ".join(df["SourceSheet"].unique())
min_date = df["InvoiceDate"].min()
max_date = df["InvoiceDate"].max()

display(Markdown(f"""
### Dataset Summary
* **Dimensions:** `{df.shape[0]:,}` rows x `{df.shape[1]}` columns
* **Source sheets:** {sheets_list}
* **Date range:** from `{min_date}` to `{max_date}`
"""))
display(df.head().style.hide(axis="index"))



### Dataset Summary
* **Dimensions:** `1,067,371` rows x `9` columns
* **Source sheets:** Year 2009-2010, Year 2010-2011
* **Date range:** from `2009-12-01 07:45:00` to `2011-12-09 12:50:00`


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.950000,13085.000000,United Kingdom,Year 2009-2010
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750000,13085.000000,United Kingdom,Year 2009-2010
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750000,13085.000000,United Kingdom,Year 2009-2010
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.100000,13085.000000,United Kingdom,Year 2009-2010
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.250000,13085.000000,United Kingdom,Year 2009-2010


**Results.** 1,067,371 rows across 9 columns, spanning two source sheets (2009-2010 and 2010-2011). The period runs from 1 December 2009 to 9 December 2011, so the final month is incomplete and any monthly trend must be read with that in mind.

## 1.2 Type Checking and Identifier Inspection

Identifiers are checked for homogeneity before any type casting or calculation.

In [3]:
display(Markdown("### Column Types"))
dtypes_df = pd.DataFrame(df.dtypes, columns=["Data Type"]).reset_index()
dtypes_df.columns = ["Column", "Type"]
display(dtypes_df.style
    .hide(axis="index"))


### Column Types

Column,Type
Invoice,string
StockCode,string
Description,string
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,string
SourceSheet,string


In [4]:
non_num_invoices = df[df["Invoice"].str.contains(r"[A-Za-z]", na=False)]
non_num_stockcodes = df[df["StockCode"].str.contains(r"[A-Za-z]", na=False)]

display(Markdown(f"* **Invoices with letters:** `{len(non_num_invoices):,}` ({len(non_num_invoices)/len(df):.2%})"))
display(Markdown(f"* **StockCodes with letters:** `{len(non_num_stockcodes):,}` ({len(non_num_stockcodes)/len(df):.2%})"))
display(Markdown("### Non-numeric characters in Invoice"))
display(non_num_invoices.head())
display(Markdown("### Non-numeric characters in StockCode"))
display(non_num_stockcodes.head())


* **Invoices with letters:** `19,500` (1.83%)

* **StockCodes with letters:** `134,986` (12.65%)

### Non-numeric characters in Invoice

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Year 2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,Year 2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Year 2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010


### Non-numeric characters in StockCode

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
12,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010
23,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078.0,United Kingdom,Year 2009-2010
28,489436,84596F,SMALL MARSHMALLOWS PINK BOWL,8,2009-12-01 09:06:00,1.25,13078.0,United Kingdom,Year 2009-2010


**Results.** 19,500 invoices (1.83%) contain letters, and 134,986 stock codes (12.65%) do. Inspecting real rows rather than the counts alone: non-numeric invoices start with `C` and carry negative quantities, which identifies cancellations. A small number start with `A`, which section 2.2 identifies as accounting adjustments. Non-numeric stock codes are mostly genuine product variants (`79323P` pink, `79323W` white), so the letter is a colour or size suffix, not a marker of a non-product line. This rules out treating letters in `StockCode` as an exclusion criterion. Cancellations are isolated as a distinct signal rather than netted here. The netting decision belongs to the analysis that consumes it, and is made explicitly in notebook 03.

## 1.3 Missing Values & Uniqueness Audit

In [5]:
line_revenue = (df["Quantity"] * df["Price"]).fillna(0.0)
total_revenue = line_revenue.sum()

n_missing = df.isna().sum()
pct_missing = (n_missing / len(df) * 100).round(2)

revenue_missing = df.isna().T.dot(line_revenue).astype(float)
pct_revenue_missing = (revenue_missing / total_revenue * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": n_missing,
    "Percentage (%)": pct_missing,
    "Revenue Missing (GBP)": revenue_missing,
    "Revenue Missing (%)": pct_revenue_missing,
}).sort_values(by="Missing Values", ascending=False)

display(Markdown("### Missing Values & Financial Impact Breakdown"))
display(missing_summary.style.format({
    "Missing Values": "{:,}",
    "Percentage (%)": "{:.2f}%",
    "Revenue Missing (GBP)": "£{:,.2f}",
    "Revenue Missing (%)": "{:.2f}%",
}))

display(Markdown(f"""### Unique Entities
* **Unique customers:** `{df['Customer ID'].nunique():,}`
* **Unique invoices:** `{df['Invoice'].nunique():,}`
* **Unique stock codes:** `{df['StockCode'].nunique():,}`
"""))

### Missing Values & Financial Impact Breakdown

,Missing Values,Percentage (%),Revenue Missing (GBP),Revenue Missing (%)
Customer ID,"243,007",22.77%,"£2,638,958.18",13.68%
Description,"4,382",0.41%,£0.00,0.00%
Invoice,0,0.00%,£0.00,0.00%
Quantity,0,0.00%,£0.00,0.00%
StockCode,0,0.00%,£0.00,0.00%
InvoiceDate,0,0.00%,£0.00,0.00%
Price,0,0.00%,£0.00,0.00%
Country,0,0.00%,£0.00,0.00%
SourceSheet,0,0.00%,£0.00,0.00%


### Unique Entities
* **Unique customers:** `5,942`
* **Unique invoices:** `53,628`
* **Unique stock codes:** `5,304`


**Results.** The raw dataset holds 5,942 unique customers, 53,628 invoices and 5,304 stock codes.`Customer ID` is missing on 22.77% of lines, which represents only 13.68% of revenue. Missing-identifier rows are therefore lower in value on average than identified ones, consistent with guest checkouts or non-attributed sales. These lines cannot contribute to segmentation, retention or prediction, and are excluded in section 3.`Description` is missing on 0.41% of lines, which is negligible and requires no rule. Measuring missingness in revenue alongside row count matters here: a rule that drops a fifth of the rows sounds severe until the revenue impact shows it drops closer to a seventh of the value.

# 2. Data Quality Assessment
## 2.1 Cancellations (`C` invoice prefix)

In [6]:
cancellations = df[df["Invoice"].str.startswith("C", na=False)]
cancellation_revenue = (cancellations["Quantity"] * cancellations["Price"]).sum()

cancellation_pct_revenue = (cancellation_revenue / total_revenue) * 100 if total_revenue != 0 else 0.0

display(Markdown(f"""
### Cancellation Summary (`C` prefix)
* **Total cancellation rows:** `{len(cancellations):,}` ({len(cancellations)/len(df):.2%})
* **Total negative quantity:** `{cancellations['Quantity'].sum():,}`
* **Net financial impact:** `£{cancellation_revenue:,.2f}` ({cancellation_pct_revenue:.2f}% of total revenue)
"""))



### Cancellation Summary (`C` prefix)
* **Total cancellation rows:** `19,494` (1.83%)
* **Total negative quantity:** `-490,992`
* **Net financial impact:** `£-1,526,667.86` (-7.92% of total revenue)


**Results.** 19,494 rows (1.83%) are cancellations, totalling -490,992 units and £-1,526,667.86, which is -7.9% of signed revenue. 
Cancelled invoices are kept, not removed. They are not successful purchases, but they carry behavioural information about returns, and the right treatment depends on the question being asked. 
A dedicated `IsCancellation` flag is created so each downstream notebook decides explicitly rather than inheriting a silent exclusion.

In [7]:
neg_qty_no_c = df[(df["Quantity"] < 0) & (~df["Invoice"].str.startswith("C", na=False))]
pos_qty_with_c = df[(df["Quantity"] > 0) & (df["Invoice"].str.startswith("C", na=False))]
zero_or_neg_price = df[df["Price"] <= 0]

display(Markdown(f"""
### Data Integrity Anomalies
* **Negative quantity without 'C':** `{len(neg_qty_no_c):,}` rows
* **Cancellation ('C') with positive quantity:** `{len(pos_qty_with_c):,}` rows
* **Zero or negative price:** `{len(zero_or_neg_price):,}` rows
"""))



### Data Integrity Anomalies
* **Negative quantity without 'C':** `3,457` rows
* **Cancellation ('C') with positive quantity:** `1` rows
* **Zero or negative price:** `6,207` rows


**Results.** Three integrity checks:

| Check | Rows | Interpretation |
|---|---|---|
| Negative quantity without a `C` prefix | 3,457 | Investigated in 2.2 || Cancellation with positive quantity | 1 | Investigated below |
| Zero or negative price | 6,207 | Investigated in 2.2 |



In [8]:
is_cancellation = (
    df["IsCancellation"]
    if "IsCancellation" in df.columns
    else df["Invoice"].astype(str).str.startswith("C", na=False)
)

anomaly = df[is_cancellation & (df["Quantity"] > 0)]
n_anomaly = len(anomaly)

if n_anomaly == 0:
    display(Markdown("### Sanity check passed\n*All cancellations carry non-positive quantities.*"))
else:
    display(Markdown(
        f"### Anomalies detected\nFound **`{n_anomaly}`** row(s) flagged as cancellations but carrying a positive `Quantity`."
    ))
    display(
        anomaly.head(10)
        .style.set_properties(**{"text-align": "left"})
        .set_table_styles([dict(selector="th", props=[("text-align", "left")])])
    )

### Anomalies detected
Found **`1`** row(s) flagged as cancellations but carrying a positive `Quantity`.

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.570000,nan,United Kingdom,Year 2009-2010


**Results.** One row (`C496350`, `StockCode = "M"`, a manual entry) is flagged as a cancellation while carrying a positive quantity. It has no `Customer ID` attached, so the section 3 exclusion removes it regardless. No dedicated rule is needed.

## 2.2 Price and Quantity Anomalies
Each anomaly identified above was checked against real rows rather than assumed.

**Negative quantities without a `C` prefix (3,457 rows).** All of them lack a `Customer ID`. The missing-identifier exclusion in section 3 absorbs them entirely, so no separate rule is required.

**Zero or negative prices (6,207 rows).** The five large negative values, up to £-53,594.36, all carry `StockCode = "B"` with `Description = "Adjust bad debt"`. This is an accounting write-off, confirmed by reading the description rather than inferred from the amount. Of the remaining zero-price rows, 98.86% have no `Customer ID` and are also absorbed by section 3.

**The 71 zero-price rows attached to a real customer** are mostly genuine products, with real descriptions and plausible quantities, supplied at no charge. These read as promotional gifts or goodwill gestures. They are kept: removing them would misrepresent what actually happened, and their revenue contribution is zero by construction.

**System test data.** Within that residual, `TEST001` and `TEST002` appeared, with `Description = "This is a test product."`, 17 rows across the full dataset at various prices rather than only zero. Customer `12346`, surfaced by these rows, was checked individually and confirmed as genuine, with real purchases across several months. Only the two test-code rows are excluded, not the customer.*

*Note on method.** The alpha-only pattern used in section 2.3 cannot match alphanumeric codes such as `TEST001`, and does not look at descriptions at all. `B`, `TEST001` and `TEST002` surfaced only by cross-checking price and quantity anomalies against real rows. Every exclusion in this notebook was resolved by inspecting actual data, not by acting on aggregate counts.

## 2.3 Non-Product Codes

`StockCode` may mix genuine products with administrative entries. This is checked rather than assumed.

In [9]:
non_product_codes = df.loc[
    df["StockCode"].str.match(r"^[A-Za-z]+$", na=False),
    "StockCode"
].value_counts()

display(Markdown("### Non-Product Codes Overview (alpha-only codes)"))
display(Markdown(f"*Found **`{len(non_product_codes)}`** unique alphabetic stock codes.*"))
npc_df = non_product_codes.reset_index()
npc_df.columns = ["StockCode", "Occurrence Count"]
display(npc_df)


### Non-Product Codes Overview (alpha-only codes)

*Found **`16`** unique alphabetic stock codes.*

,StockCode,Occurrence Count
0,POST,2122
1,DOT,1446
2,M,1421
3,D,177
4,S,104
5,ADJUST,67
6,AMAZONFEE,43
7,DCGSSGIRL,25
8,DCGSSBOY,23
9,PADS,19


**Results.** 16 unique alphabetic stock codes, dominated by `POST` (2,122 rows), `DOT` (1,446) and `M` (1,421).The filter over-catches genuine products. `DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL` and `DCGSLBOY` form a childrenswear line, `PADS` is a sellable item, and `GIFT` is a real product. All are kept. `m` is merged with `M` case-insensitively. The filter also misses `TEST001` and `TEST002`, which are alphanumeric and surfaced only through section 2.2. Confirmed non-product codes, each verified against its description or transaction pattern:

| Code | Meaning | Evidence |
|---|---|---|
| `POST` | Postage | Value counts, plausible volume || `DOT` | Dotcom postage charge | Value counts |
| `M`, `m` | Manual entry | Value counts, merged case-insensitively || `D` | Discount | Value counts |
| `S` | Samples | Value counts || `ADJUST` | Accounting adjustment | Description, for example "Adjustment by john on..." || `AMAZONFEE` | Amazon platform fee | Value counts, large negative average || `CRUK` | Cancer Research UK donation | Value counts || `B` | Bad debt adjustment | Description "Adjust bad debt" || `TEST001`, `TEST002` | System test data | Description "This is a test product." |

In [10]:
NON_PRODUCT_CODES = {
    "POST": "Postage",
    "DOT": "Dotcom postage / charge",
    "M": "Manual entry",
    "D": "Discount",
    "S": "Samples",
    "ADJUST": "Accounting adjustment",
    "AMAZONFEE": "Amazon platform fee",
    "CRUK": "Cancer Research UK donation",
    "B": "Bad debt adjustment",
    "TEST001": "Test product (system test data)",
    "TEST002": "Test product (system test data)",
}

df["StockCodeUpper"] = df["StockCode"].str.upper()
df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES.keys())

non_product_df = df[df["IsNonProduct"]]
non_product_summary = (
    non_product_df.groupby("StockCodeUpper")
    .apply(
        lambda g: pd.Series({
            "Description": NON_PRODUCT_CODES.get(g.name, "Other"),
            "Total Rows": len(g),
            "Total Quantity": g["Quantity"].sum(),
            "Total Revenue (GBP)": (g["Quantity"] * g["Price"]).sum(),
        }),
        include_groups=False,
    )
    .reset_index()
)

display(Markdown("### Non-Product Codes Overview (final)"))
display(non_product_summary.style.format({
    "Total Rows": "{:,}",
    "Total Quantity": "{:,}",
    "Total Revenue (GBP)": "£{:,.2f}",
}))


### Non-Product Codes Overview (final)

,StockCodeUpper,Description,Total Rows,Total Quantity,Total Revenue (GBP)
0,ADJUST,Accounting adjustment,67,5,"£6,835.24"
1,AMAZONFEE,Amazon platform fee,43,-35,"£-260,763.58"
2,B,Bad debt adjustment,6,6,"£-147,614.08"
3,CRUK,Cancer Research UK donation,16,-16,"£-7,933.43"
4,D,Discount,177,"-2,872","£-13,484.54"
5,DOT,Dotcom postage / charge,"1,446","2,938","£322,647.47"
6,M,Manual entry,"1,426","4,612","£-82,781.27"
7,POST,Postage,"2,122","10,108","£112,341.00"
8,S,Samples,104,-98,"£-6,065.80"
9,TEST001,Test product (system test data),15,55,£202.50


**Results.** These lines carry no product or customer-behaviour information and are excluded from the analytical base. They remain in the raw table, flagged through `IsNonProduct` rather than deleted, so the exclusion stays auditable.

# 3. Building the Analytical Base
## 3.1 Filtering & Feature Engineering

Two exclusions are applied together: 

1. Non-product lines (`IsNonProduct`), including `B`, `TEST001` and `TEST002` confirmed in section 2.2.
2. Rows without a `Customer ID`.
3. Cancellations are kept and flagged. Whether to net them into a customer's monetary value or treat cancellation behaviour as a separate signal is decided in notebook 03.

In [11]:
initial_rows = len(df)
total_raw_revenue = (df["Quantity"] * df["Price"]).sum()

if "StockCodeUpper" not in df.columns:
    df["StockCodeUpper"] = df["StockCode"].str.upper()
if "IsNonProduct" not in df.columns:
    df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES.keys())
df["IsCancellation"] = df["Invoice"].str.startswith("C", na=False)

step1 = df.loc[~df["IsNonProduct"]]
rows_after_non_product = len(step1)

df_clean = step1.loc[step1["Customer ID"].notna()].copy()
rows_final = len(df_clean)

df_clean["Customer ID"] = df_clean["Customer ID"].astype("int64").astype("string")
df_clean["LineRevenue"] = df_clean["Quantity"] * df_clean["Price"]

pct_clean = (rows_final / initial_rows) * 100
n_unique_cust = df_clean["Customer ID"].nunique()
revenue_retained = df_clean["LineRevenue"].sum()
pct_revenue_retained = (revenue_retained / total_raw_revenue) * 100
n_cancellations_kept = df_clean["IsCancellation"].sum()

display(Markdown(f"""
### Cleaned Dataset Ready
* **Analytical base:** `{rows_final:,}` rows (`{pct_clean:.2f}%` of raw data)
* **Unique customers:** `{n_unique_cust:,}`
* **Revenue retained:** `£{revenue_retained:,.2f}` (`{pct_revenue_retained:.2f}%` of raw total revenue)
* **Cancellation rows kept (flagged):** `{n_cancellations_kept:,}`
"""))



### Cleaned Dataset Ready
* **Analytical base:** `820,947` rows (`76.91%` of raw data)
* **Unique customers:** `5,880`
* **Revenue retained:** `£16,728,372.22` (`86.73%` of raw total revenue)
* **Cancellation rows kept (flagged):** `17,946`


**Results.** The analytical base holds 820,947 rows, 76.91% of the raw data, covering 5,880 unique customers and £16,728,372.22, which is 86.73% of raw revenue. 17,946 cancellation rows are retained and flagged.The asymmetry between rows and revenue is the point: excluding 23% of rows costs only 13% of revenue, because unidentified transactions are systematically smaller.

## 3.2 Export

The prepared dataset is written to Parquet so every downstream notebook loads identical preprocessing.

In [12]:
out_path = PROC / "clean_transactions.parquet"
df_clean.to_parquet(out_path, index=False)
display(Markdown(f"Saved successfully to: `{out_path}`"))


Saved successfully to: `..\data\processed\clean_transactions.parquet`

# 4. Data Preparation Report
## 4.1 Impact Quantification

In [13]:
decision_log = pd.DataFrame([
    {"Step": "Raw data", "Condition": "Initial dataset",
     "Rows Remaining": initial_rows, "Rows Removed": 0, "% Retained": "100.0%"},
    {"Step": "1. Non-product codes",
     "Condition": "Remove POST, DOT, M, D, S, ADJUST, AMAZONFEE, CRUK, B, TEST001, TEST002",
     "Rows Remaining": rows_after_non_product,
     "Rows Removed": initial_rows - rows_after_non_product,
     "% Retained": f"{rows_after_non_product/initial_rows:.1%}"},
    {"Step": "2. Missing Customer ID", "Condition": "Remove rows without a Customer ID",
     "Rows Remaining": rows_final, "Rows Removed": rows_after_non_product - rows_final,
     "% Retained": f"{rows_final/initial_rows:.1%}"},
])
display(Markdown("### Cleaning Decision Log & Impact Summary"))
display(decision_log)


### Cleaning Decision Log & Impact Summary

,Step,Condition,Rows Remaining,Rows Removed,% Retained
0,Raw data,Initial dataset,1067371,0,100.0%
1,1. Non-product codes,"Remove POST, DOT, M, D, S, ADJUST, AMAZONFEE, ...",1061947,5424,99.5%
2,2. Missing Customer ID,Remove rows without a Customer ID,820947,241000,76.9%


**Results.** Non-product codes remove 5,424 rows (0.5%), and missing customer identifiers remove a further 241,000 rows (22.6%). 76.9% of the raw data reaches the analytical base.

## 4.2 Decision Log

| # | Decision | Notes |
|---|---|---|
| 1 | Non-product codes excluded | `POST`, `DOT`, `M`, `D`, `S`, `ADJUST`, `AMAZONFEE`, `CRUK`, `B`, `TEST001`, `TEST002`, each verified against its description or value counts |
| 2 | Missing Customer ID excluded | Absorbs 100% of negative-quantity anomalies and 98.86% of zero-price anomalies from section 2.2 |
| 3 | Cancellations flagged and kept | Netting decision deferred to notebook 03 |
| 4 | 71 zero-price rows with a real customer kept | Genuine promotional gifts, not anomalies |
| 5 | Alphabetic stock codes kept when genuine | `DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL`, `DCGSLBOY`, `PADS`, `GIFT` |

## 4.3 Limitations
**Guest transactions are out of scope.** 22.77% of rows carry no customer identifier and are excluded, so every customer-level metric downstream describes identified customers only. Any inference about the full customer base would be biased toward higher-value, repeat buyers.
**The final month is incomplete.
** The dataset stops on 9 December 2011, so December 2011 figures are not comparable to earlier months.
**Country is recorded per transaction, not per customer.
** A customer ordering from two countries appears under both, which notebook 02 resolves explicitly.## 4.4 Summary`clean_transactions.parquet` is the single source for notebooks 02 to 05: 820,947 rows, 5,880 customers, £16.73M in net revenue, cancellations preserved and flagged.Every exclusion was resolved by inspecting real rows rather than acting on aggregate counts. Three codes (`B`, `TEST001`, `TEST002`) would have escaped the alpha-only filter used in section 2.3 and were found only through the price and quantity investigation in section 2.2.